### Installation

In [1]:
%%capture
!pip install unsloth  # Do this in local & cloud setups
!pip install langchain-text-splitters langchain-community pymupdf

In [2]:
import unsloth
from langchain_community.document_loaders import PyMuPDFLoader
import pandas as pd
from tqdm import tqdm
from transformers import TextStreamer
from unsloth import FastLanguageModel
import torch

/tmp/ipykernel_10627/507993306.py:5: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
fourbit_models = [
    "unsloth/Qwen3-1.7B-unsloth-bnb-4bit", # Qwen 14B 2x faster
    "unsloth/Qwen3-4B-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    "unsloth/Qwen3-14B-unsloth-bnb-4bit",
    "unsloth/Qwen3-32B-unsloth-bnb-4bit",

    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/Phi-4",
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/orpheus-3b-0.1-ft-unsloth-bnb-4bit" # [NEW] We support TTS models!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-14B",
    max_seq_length = 2048,   # Context length - can be longer, but uses more memory
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    # token = "YOUR_HF_TOKEN",      # HF Token for gated models
)

==((====))==  Unsloth 2026.5.2: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/443 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/qwen3-14b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,  # Best to choose alpha = rank or rank*2
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

Unsloth 2026.5.2 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


In [5]:
loader = PyMuPDFLoader('https://eaa-online.org/app/uploads/sites/80/2026/05/scientific_programme-open.pdf')
docs = loader.load()

data = pd.DataFrame()
for i in range(len(docs)):
    data.at[i, 'page'] = int(docs[i].metadata['page'])
    data.at[i, 'text'] = docs[i].page_content

len(data)

111

In [6]:
system_prompt = {
    'name': 'system_prompt_v2', 'text':'''
    Exctract category (usually menstioned on the top of the page), paper name, authors, and universities for every paper from the provided text.
    Output format:
    \n[Category]:[category];[Name]:[paper name];[Author_1]:[name of the 1st author];[Uni_1]:[university name of the 1st author];[Author_2]:[name of the 2nd author if exists];[Uni_2]:[university name of the 2nd author if exists];[Author_3]:[name of the 3rd author if exists];[Uni_3]:[university name of the 3rd author if exists];[Author_4]:[name of the 4th author if exists];[Uni_4]:[university name of the 4th author if exists];[Author_5]:[name of the 5th author if exists];[Uni_5]:[university name of the 5th author if exists]\n
'''
}

In [7]:
MAX_NEW_TOKENS = 2048
TEMPERATURE = 1
TOP_P = 0.95
TOP_K = 20

In [8]:
results = pd.DataFrame()
results['page'] = data['page']
results['text'] = data['text']

In [9]:
results['results'] = None
for i, ro in tqdm(results.iterrows(), total=len(data)):
    text = ro['text']

    # handling empty text
    if pd.isna(text) or not text.strip():
      continue

    messages = [{"role": "system", "content": system_prompt['text']},
                {"role": "user", "content": f"{text}"}]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # Must add for generation
        enable_thinking=False,        # Enable thinking
    )

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

    generated_ids = model.generate(
    **inputs,
    max_new_tokens = MAX_NEW_TOKENS,           # Increase for longer outputs!
    temperature = TEMPERATURE,
    top_p = TOP_P,
    top_k = TOP_K,
    do_sample = True,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
    )

    generated = tokenizer.decode(
    generated_ids[0][inputs.input_ids.shape[1]:],
    skip_special_tokens=True
    )

    results.at[i, 'results'] = generated
results.to_csv(f'results.csv')

  0%|          | 0/111 [00:00<?, ?it/s]Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, 

[Category]:AU;[Name]:;[Author_1]:;[Uni_1]:;[Author_2]:;[Uni_2]:;[Author_3]:;[Uni_3]:;[Author_4]:;[Uni_4]:;[Author_5]:;[Uni_5]:  
[Category]:ED;[Name]:;[Author_1]:;[Uni_1]:;[Author_2]:;[Uni_2]:;[Author_3]:;[Uni_3]:;[Author_4]:;[Uni_4]:;[Author_5]:;[Uni_5]:  
[Category]:FA;[Name]:;[Author_1]:;[Uni_1]:;[Author_2]:;[Uni_2]:;[Author_3]:;[Uni_3]:;[Author_4]:;[Uni_4]:;[Author_5]:;[Uni_5]:  
[Category]:FR;[Name]:;[Author_1]:;[Uni_1]:;[Author_2]:;[Uni_2]:;[Author_3]:;[Uni_3]:;[Author_4]:;[Uni_4]:;[Author_5]:;[Uni_5]:  
[Category]:GV;[Name]:;[Author_1]:;[Uni_1]:;[Author_2]:;[Uni_2]:;[Author_3]:;[Uni_3]:;[Author_4]:;[Uni_4]:;[Author_5]:;[Uni_5]:  
[Category]:HI;[Name]:;[Author_1]:;[Uni_1]:;[Author_2]:;[Uni_2]:;[Author_3]:;[Uni_3]:;[Author_4]:;[Uni_4]:;[Author_5]:;[Uni_5]:  
[Category]:IC;[Name]:;[Author_1]:;[Uni_1]:;[Author_2]:;[Uni_2]:;[Author_3]:;[Uni_3]:;[Author_4]:;[Uni_4]:;[Author_5]:;[Uni_5]:  
[Category]:IS;[Name]:;[Author_1]:;[Uni_1]:;[Author_2]:;[Uni_2]:;[Author_3]:;[Uni_3]:;[Author_4]:;

  1%|          | 1/111 [01:56<3:32:50, 116.10s/it]Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Category]:Auditing;[Name]:Accounting scandals, auditor ratification, and institutional investors: evidence from the Wirecard scandal in Germany;[Author_1]:Tessa Kunkel;[Uni_1]:ESCP Business School;[Author_2]:;[Uni_2]:;[Author_3]:;[Uni_3]:;[Author_4]:;[Uni_4]:;[Author_5]:;[Uni_5]:  
[Category]:Auditing;[Name]:Media framing and the transformation of the UK audit regulator after Carillion;[Author_1]:Alan Duboisee de Ricquebourg;[Uni_1]:Leeds University Business School;[Author_2]:Warren Maroun;[Uni_2]:Leeds University Business School;[Author_3]:Dannielle Cerbone;[Uni_3]:University of the Witwatersrand;[Author_4]:Wayne van Zijl;[Uni_4]:University of the Witwatersrand;[Author_5]:;[Uni_5]:  
[Category]:Auditing;[Name]:Navigating credibility in unregulated assurance markets;[Author_1]:Conor Clune;[Uni_1]:UNSW Sydney;[Author_2]:;[Uni_2]:;[Author_3]:;[Uni_3]:;[Author_4]:;[Uni_4]:;[Author_5]:;[Uni_5]:  
[Category]:Auditing;[Name]:Do assigned audit partners perform higher quality audits than self

  2%|▏         | 2/111 [03:56<3:35:06, 118.40s/it]Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

 98%|█████████▊| 109/111 [3:04:43<03:29, 104.59s/it]Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Category]:Taxation;[Name]:Environmental-unfriendly tax avoidance;[Author_1]:Zhimin Chen;[Uni_1]:Nanyang Business School;[Author_2]:Martin Jacob;[Uni_2]:IESE Business School;[Author_3]:Xiang Zheng;[Uni_3]:Nanyang Technological University  
[Category]:Taxation;[Name]:Carbon pricing and cross-border innovation spillovers;[Author_1]:Cinthia Valle Ruiz;[Uni_1]:IE University;[Author_2]:Martin Jacob;[Uni_2]:IESE Business School;[Author_3]:Christof Beuselinck;[Uni_3]:IESEG School of Management  
[Category]:Taxation;[Name]:Personal income taxes and corporate pollution;[Author_1]:Michael Mayberry;[Uni_1]:University of Florida;[Author_2]:Marvin Nipper;[Uni_2]:Friedrich-Alexander-Universität Erlangen-Nürnberg;[Author_3]:Marius Weiß;[Uni_3]:Friedrich-Alexander-Universität Erlangen-Nürnberg  
[Category]:Taxation;[Name]:(Mis)measurement of income shifting;[Author_1]:Stefanie Pendl;[Uni_1]:Vienna University of Economics and Business;[Author_2]:Harald Amberger;[Uni_2]:Vienna University of Economics an

 99%|█████████▉| 110/111 [3:06:06<01:38, 98.14s/it] Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Category]:Taxation;[Name]:Tax advisory firms and reputational costs: client purchases of auditor-provided tax services after a tax scandal;[Author_1]:Kenny Dekoster;[Uni_1]:Ghent University;[Author_2]:Inga Hardeck;[Uni_2]:University of Duisburg-Essen;[Author_3]:Beatrice Renges;[Uni_3]:University of Duisburg-Essen;[Author_4]:Isabelle Verleyen;[Uni_4]:Ghent University;[Author_5]:;[Uni_5]:  
[Category]:Taxation;[Name]:Tax administration compliance burdens: global measurement and evidence;[Author_1]:Jesse Marangoni;[Uni_1]:Tilburg University;[Author_2]:;[Uni_2]:;[Author_3]:;[Uni_3]:;[Author_4]:;[Uni_4]:;[Author_5]:;[Uni_5]:<|im_end|>


100%|██████████| 111/111 [3:06:33<00:00, 100.85s/it]
